# UKRI FoR Classifier — POC deployment notebook
## Primary Group model + fallback Division model

This version reflects the latest agreed behaviour:

- Input and final output are **Parquet**.
- `ApplicationID + ApplicationOriginSource` is the application identity.
- Duplicate input applications are removed (first occurrence retained) and saved locally for audit.
- Only `ApplicationTitle` and `ApplicationSummary` are used for inference.
- If both text fields are null/blank, the application is **not sent to either model but remains in final output**.
- Primary model predicts 4-digit FoR **Group** codes.
- Only primary-unresolved records go to the fallback model, which predicts 2-digit FoR **Division** codes.
- If neither model classifies an application, the final output still includes it with native nulls in `category_id`, `score_type`, and `score`.
- `category_id` is a nullable string/varchar-compatible field.
- `model_run_date` is a full datetime timestamp.
- BACKFILL mode processes the initial 10 files independently.
- DELTA mode selects the latest timestamped **unprocessed** file.
- Processed-file state is stored locally in Ronin at `control/processed_files.json`; nothing is written to input S3 for control purposes.

## 1. Imports and project paths

In [ ]:
from pathlib import Path
from datetime import datetime
import json, logging, re, sys
import boto3, joblib, numpy as np, pandas as pd

CURRENT_DIR = Path.cwd()
if (CURRENT_DIR / 'models').exists():
    PROJECT_ROOT = CURRENT_DIR
elif (CURRENT_DIR.parent / 'models').exists():
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    PROJECT_ROOT = CURRENT_DIR.parent

DATA_DIR = PROJECT_ROOT / 'data'
INPUT_DIR = DATA_DIR / 'input'
OUTPUT_DIR = DATA_DIR / 'output'
DUPLICATE_DIR = DATA_DIR / 'duplicates'
CONTROL_DIR = PROJECT_ROOT / 'control'
LOG_DIR = PROJECT_ROOT / 'logs'
for d in [INPUT_DIR, OUTPUT_DIR, DUPLICATE_DIR, CONTROL_DIR, LOG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
from data_preparation import preprocess_text_fields

print('PROJECT_ROOT:', PROJECT_ROOT)

## 2. Configuration — review before each run

In [ ]:
# RUN MODE: 'BACKFILL' or 'DELTA'
RUN_MODE = 'BACKFILL'

# S3
S3_BUCKET = 'ddg-poc-data.store.stg-ukri.cloud'
S3_INPUT_PREFIX = 'deployment_test_data/'
S3_OUTPUT_PREFIX = 'test_output/'
INPUT_SUFFIXES = ('.parquet',)
OUTPUT_EXTENSION = '.parquet'

# BACKFILL: populate exact keys if the prefix contains more than the 10 initial files.
BACKFILL_KEYS = []
BACKFILL_EXPECTED_FILE_COUNT = 10

# Input filenames begin with yyyymmddhhmm_
FILENAME_TIMESTAMP_FORMAT = '%Y%m%d%H%M'

# Local Ronin state (not S3)
PROCESSING_MANIFEST_PATH = CONTROL_DIR / 'processed_files.json'

# Keep True for the real end-to-end run. Set False while validating locally.
UPLOAD_OUTPUT_TO_S3 = True

# Models
MAIN_MODEL_PATH = PROJECT_ROOT / 'models' / (
    'lemma_stop_neg_scale_10_mindf_5_maxdf_90_maxfold_5_min_pos_50_pos_prior_50_'
    'fields_of_research_negative_scaling_tfidf.joblib'
)
FALLBACK_MODEL_PATH = PROJECT_ROOT / 'models' / (
    'lemma_stop_neg_scale_10_mindf_5_maxdf_90_maxfold_5_min_pos_50_pos_prior_50_'
    'fields_of_research_negative_scaling_tfidf_division.joblib'
)

ID_FIELDS = ['ApplicationID', 'ApplicationOriginSource']
MODEL_TEXT_FIELDS = ['ApplicationTitle', 'ApplicationSummary']
TAXONOMY_FILE_TOKEN = 'FoR'
TAXONOMY_VALUE = 'FieldsOfResearch'
MODEL_NAME = 'FoRClassification'
MODEL_VERSION = '1.1'
SCORE_TYPE = 'uncalibrated'
N_JOBS = 4
PREPROCESS_BATCH_SIZE = 250

print('RUN_MODE:', RUN_MODE)
print('MAIN_MODEL_PATH:', MAIN_MODEL_PATH)
print('FALLBACK_MODEL_PATH:', FALLBACK_MODEL_PATH)
print('MANIFEST:', PROCESSING_MANIFEST_PATH)

## 3. Runtime checks and logging

In [ ]:
if RUN_MODE not in {'BACKFILL','DELTA'}:
    raise ValueError("RUN_MODE must be 'BACKFILL' or 'DELTA'")
for p in [MAIN_MODEL_PATH, FALLBACK_MODEL_PATH]:
    if not p.exists():
        raise FileNotFoundError(p)
try:
    import pyarrow
except ImportError as exc:
    raise ImportError('pyarrow is required for Parquet I/O') from exc

session_run_timestamp = datetime.now()
session_run_stamp = session_run_timestamp.strftime('%Y%m%d_%H%M%S')
log_path = LOG_DIR / f'for_inference_{session_run_stamp}.log'
logger = logging.getLogger('for_inference')
logger.setLevel(logging.INFO)
logger.handlers.clear()
fmt = logging.Formatter('%(asctime)s | %(levelname)s | %(message)s')
fh = logging.FileHandler(log_path); fh.setFormatter(fmt); logger.addHandler(fh)
sh = logging.StreamHandler(); sh.setFormatter(fmt); logger.addHandler(sh)
logger.info('Started notebook in %s mode', RUN_MODE)

## 4. Local processed-file manifest

In [ ]:
def load_manifest(path):
    if not path.exists():
        return {'processed_files': []}
    with path.open('r', encoding='utf-8') as f:
        data=json.load(f)
    if not isinstance(data.get('processed_files'), list):
        raise ValueError(f'Invalid manifest: {path}')
    return data

def save_manifest(path, data):
    tmp=path.with_suffix('.tmp')
    with tmp.open('w', encoding='utf-8') as f:
        json.dump(data, f, indent=2)
    tmp.replace(path)

def processed_keys(data):
    return {x['input_key'] for x in data['processed_files'] if x.get('status')=='SUCCESS' and x.get('input_key')}

def mark_processed(data, input_key, output_key, output_rows, unique_apps):
    data['processed_files']=[x for x in data['processed_files'] if x.get('input_key') != input_key]
    data['processed_files'].append({
        'input_key': input_key,
        'processed_at': datetime.now().isoformat(timespec='seconds'),
        'output_key': output_key,
        'status':'SUCCESS',
        'output_rows': int(output_rows),
        'unique_applications': int(unique_apps),
        'model_name': MODEL_NAME,
        'model_version': MODEL_VERSION,
    })
    return data

processing_manifest=load_manifest(PROCESSING_MANIFEST_PATH)
print('Previously processed:', len(processed_keys(processing_manifest)))

## 5. Discover eligible Parquet files and select BACKFILL/DELTA inputs

In [ ]:
s3=boto3.client('s3')

def parse_filename_datetime(key):
    token=Path(key).name.split('_',1)[0]
    return datetime.strptime(token, FILENAME_TIMESTAMP_FORMAT)

def discover_files():
    paginator=s3.get_paginator('list_objects_v2')
    rows=[]
    for page in paginator.paginate(Bucket=S3_BUCKET, Prefix=S3_INPUT_PREFIX):
        for obj in page.get('Contents',[]):
            key=obj['Key']
            if not key.lower().endswith(INPUT_SUFFIXES):
                continue
            try:
                dt=parse_filename_datetime(key)
            except ValueError:
                logger.warning('Ignoring file without supported timestamp: %s', key)
                continue
            rows.append({'key':key,'file_datetime':dt,'last_modified':obj['LastModified'],'size':obj['Size']})
    return sorted(rows,key=lambda x:x['file_datetime'])

discovered_files=discover_files()
print('Eligible files discovered:', len(discovered_files))

def select_files_to_process():
    by_key={x['key']:x for x in discovered_files}
    if RUN_MODE=='BACKFILL':
        if BACKFILL_KEYS:
            missing=[k for k in BACKFILL_KEYS if k not in by_key]
            if missing:
                raise FileNotFoundError(f'BACKFILL keys not found: {missing}')
            keys=list(BACKFILL_KEYS)
        else:
            keys=[x['key'] for x in discovered_files]
            if len(keys) != BACKFILL_EXPECTED_FILE_COUNT:
                raise ValueError(
                    f'Expected exactly {BACKFILL_EXPECTED_FILE_COUNT} backfill files, found {len(keys)}. '
                    'Populate BACKFILL_KEYS explicitly if the prefix also contains delta/history files.'
                )
        return sorted(keys,key=parse_filename_datetime)

    unprocessed=[x for x in discovered_files if x['key'] not in processed_keys(processing_manifest)]
    if not unprocessed:
        raise RuntimeError('No unprocessed eligible Parquet files found for DELTA mode')
    latest=max(unprocessed,key=lambda x:x['file_datetime'])
    return [latest['key']]

files_to_process=select_files_to_process()
print('Selected files:', len(files_to_process))
for k in files_to_process: print(' -',k)

## 6. Load and validate both single-joblib model bundles

In [ ]:
def load_bundle(path):
    bundle=joblib.load(path)
    if not isinstance(bundle,dict):
        raise TypeError(f'{path.name} must contain a dict')
    required={'vectorizer','models','thresholds','mlb'}
    missing=required-set(bundle)
    if missing:
        raise KeyError(f'{path.name} missing {sorted(missing)}')
    thresholds=np.asarray(bundle['thresholds']).reshape(-1)
    if not (len(bundle['models'])==len(thresholds)==len(bundle['mlb'].classes_)):
        raise ValueError(f'Incompatible bundle lengths in {path.name}')
    bundle=dict(bundle); bundle['thresholds']=thresholds
    return bundle

primary_model=load_bundle(MAIN_MODEL_PATH)
fallback_model=load_bundle(FALLBACK_MODEL_PATH)
print('Primary classes:',len(primary_model['mlb'].classes_))
print('Fallback classes:',len(fallback_model['mlb'].classes_))

## 7. Reusable inference helpers

In [ ]:
def positive_probability(model,X):
    p=np.asarray(model.predict_proba(X))
    if p.ndim==1: return p
    if p.shape[1]==1: return p[:,0]
    return p[:,1]

def perform_inference(bundle,df_in):
    n_classes=len(bundle['mlb'].classes_)
    if len(df_in)==0:
        return np.empty((0,n_classes)), np.empty((0,n_classes),dtype=np.int8)
    X=bundle['vectorizer'].transform(df_in['PROCESSED_TEXT'])
    probs=np.column_stack([positive_probability(m,X) for m in bundle['models']])
    preds=(probs >= bundle['thresholds'].reshape(1,-1)).astype(np.int8)
    if probs.shape != (len(df_in),n_classes):
        raise ValueError(f'Unexpected probability shape {probs.shape}')
    return probs,preds

def category_code(label,digits):
    m=re.match(rf'^\s*(\d{{{digits}}})\b',str(label))
    if not m:
        raise ValueError(f'Cannot extract {digits}-digit code from {label!r}')
    return m.group(1)  # string intentionally

def predictions_to_long(source_df,probs,preds,bundle,digits,source_name):
    rows=[]; source_df=source_df.reset_index(drop=True)
    for row_pos, row in source_df.iterrows():
        for class_idx in np.flatnonzero(preds[row_pos]==1):
            rows.append({
                'ApplicationID':row['ApplicationID'],
                'ApplicationOriginSource':row['ApplicationOriginSource'],
                'category_id':category_code(bundle['mlb'].classes_[class_idx],digits),
                'score':float(probs[row_pos,class_idx]),
                'prediction_source':source_name,
            })
    return pd.DataFrame(rows,columns=ID_FIELDS+['category_id','score','prediction_source'])

## 8. Process one Parquet input file end-to-end

In [ ]:
def process_one_file(input_key):
    global processing_manifest
    started=datetime.now()
    logger.info('Processing %s',input_key)

    # A. Download/read Parquet
    local_input=INPUT_DIR / Path(input_key).name
    s3.download_file(S3_BUCKET,input_key,str(local_input))
    df_raw=pd.read_parquet(local_input,engine='pyarrow')
    required=ID_FIELDS+MODEL_TEXT_FIELDS
    missing=[c for c in required if c not in df_raw.columns]
    if missing: raise ValueError(f'{input_key} missing {missing}')

    # B. Deduplicate on composite application identity; keep first
    duplicate_mask=df_raw.duplicated(subset=ID_FIELDS,keep='first')
    df_duplicates=df_raw.loc[duplicate_mask].copy()
    if len(df_duplicates):
        dup_path=DUPLICATE_DIR / f'{Path(input_key).stem}_duplicates.parquet'
        df_duplicates.to_parquet(dup_path,index=False,engine='pyarrow')
    df_base=df_raw.drop_duplicates(subset=ID_FIELDS,keep='first').reset_index(drop=True)
    df_output_base=df_base[ID_FIELDS].copy()  # MASTER population

    # C. Exclude both-null/blank text ONLY from model inference, not final output
    def blank(s): return s.isna() | s.astype('string').fillna('').str.strip().eq('')
    no_text_mask=blank(df_base['ApplicationTitle']) & blank(df_base['ApplicationSummary'])
    df_model_input=df_base.loc[~no_text_mask].copy().reset_index(drop=True)

    # D. Shared preprocessing
    if len(df_model_input):
        df_cleaned=preprocess_text_fields(
            df=df_model_input.copy(), text_fields=MODEL_TEXT_FIELDS,
            new_field_name='PROCESSED_TEXT', n_jobs=N_JOBS,
            batch_size=PREPROCESS_BATCH_SIZE,
        )
        if 'PROCESSED_TEXT' not in df_cleaned.columns:
            raise KeyError('PROCESSED_TEXT not created')
    else:
        df_cleaned=df_model_input.copy(); df_cleaned['PROCESSED_TEXT']=pd.Series(dtype='string')

    # E. Primary Group model
    p_probs,p_preds=perform_inference(primary_model,df_cleaned)
    p_counts=p_preds.sum(axis=1) if len(df_cleaned) else np.array([],dtype=int)
    p_resolved=p_counts>0
    df_primary=predictions_to_long(df_cleaned,p_probs,p_preds,primary_model,4,'PRIMARY_GROUP')

    # F. Fallback Division model ONLY on primary-unresolved rows
    df_unresolved=df_cleaned.loc[~p_resolved].copy().reset_index(drop=True)
    f_probs,f_preds=perform_inference(fallback_model,df_unresolved)
    f_counts=f_preds.sum(axis=1) if len(df_unresolved) else np.array([],dtype=int)
    df_fallback=predictions_to_long(df_unresolved,f_probs,f_preds,fallback_model,2,'FALLBACK_DIVISION')

    # G. Merge all predictions onto complete unique input population
    df_predictions=pd.concat([df_primary,df_fallback],ignore_index=True)
    df_output=df_output_base.merge(df_predictions,on=ID_FIELDS,how='left',validate='one_to_many')

    # H. Stakeholder fields + native null policy
    df_output['model_run_date']=pd.Timestamp.now()
    prediction_exists=df_output['category_id'].notna()
    df_output['score_type']=pd.Series(pd.NA,index=df_output.index,dtype='string')
    df_output.loc[prediction_exists,'score_type']=SCORE_TYPE
    df_output['category_id']=df_output['category_id'].astype('string')
    df_output['score']=pd.to_numeric(df_output['score'],errors='coerce').round(2).astype('Float64')
    df_output['Taxonomy']=TAXONOMY_VALUE
    df_output['model_name']=MODEL_NAME
    df_output['model_version']=MODEL_VERSION

    # Explicitly keep all three prediction-result columns natively null when unresolved/no text
    null_pred=df_output['category_id'].isna()
    df_output.loc[null_pred,'category_id']=pd.NA
    df_output.loc[null_pred,'score_type']=pd.NA
    df_output.loc[null_pred,'score']=pd.NA

    FINAL_COLUMNS=[
        'ApplicationID','ApplicationOriginSource','model_run_date','category_id',
        'score_type','score','Taxonomy','model_name','model_version'
    ]
    df_output=df_output[FINAL_COLUMNS]
    for c in ['ApplicationID','ApplicationOriginSource','category_id','score_type','Taxonomy','model_name','model_version']:
        df_output[c]=df_output[c].astype('string')

    # I. Completeness/quality checks
    unique_input=len(df_output_base)
    unique_output=df_output[ID_FIELDS].drop_duplicates().shape[0]
    if unique_input != unique_output:
        raise ValueError(f'Completeness failure: input={unique_input}, output={unique_output}')
    predicted=df_output['category_id'].notna()
    if df_output.loc[predicted,'score_type'].isna().any() or df_output.loc[predicted,'score'].isna().any():
        raise ValueError('Predicted rows contain null score_type/score')
    unresolved=df_output['category_id'].isna()
    if df_output.loc[unresolved,['category_id','score_type','score']].notna().any().any():
        raise ValueError('Unresolved rows must have category_id, score_type and score all null')
    if df_output.loc[predicted].duplicated(subset=ID_FIELDS+['category_id']).any():
        raise ValueError('Duplicate application/category prediction rows found')
    if df_output.loc[unresolved].duplicated(subset=ID_FIELDS).any():
        raise ValueError('Duplicate unresolved application rows found')

    # J. Local Parquet and round-trip check
    out_stamp=datetime.now().strftime(FILENAME_TIMESTAMP_FORMAT)
    output_filename=f'{out_stamp}_{TAXONOMY_FILE_TOKEN}{OUTPUT_EXTENSION}'
    local_output=OUTPUT_DIR / output_filename
    df_output.to_parquet(local_output,index=False,engine='pyarrow')
    rt=pd.read_parquet(local_output,engine='pyarrow')
    if len(rt)!=len(df_output): raise ValueError('Parquet round-trip row-count mismatch')

    # K. S3 upload; only then update local manifest
    output_key=f"{S3_OUTPUT_PREFIX.rstrip('/')}/{output_filename}"
    uploaded=False
    if UPLOAD_OUTPUT_TO_S3:
        s3.upload_file(str(local_output),S3_BUCKET,output_key)
        head=s3.head_object(Bucket=S3_BUCKET,Key=output_key)
        if int(head['ContentLength'])<=0: raise RuntimeError('Uploaded output is empty')
        uploaded=True
        processing_manifest=mark_processed(
            processing_manifest,input_key,output_key,len(df_output),unique_output
        )
        save_manifest(PROCESSING_MANIFEST_PATH,processing_manifest)

    result={
        'input_key':input_key,
        'raw_input_rows':int(len(df_raw)),
        'duplicate_rows_removed':int(len(df_duplicates)),
        'unique_input_applications':int(unique_input),
        'both_text_fields_missing':int(no_text_mask.sum()),
        'primary_resolved_applications':int(p_resolved.sum()),
        'sent_to_fallback':int(len(df_unresolved)),
        'fallback_resolved_applications':int((f_counts>0).sum()) if len(f_counts) else 0,
        'still_unresolved_after_fallback':int((f_counts==0).sum()) if len(f_counts) else 0,
        'final_null_prediction_applications':int(df_output.loc[unresolved,ID_FIELDS].drop_duplicates().shape[0]),
        'final_output_rows':int(len(df_output)),
        'final_unique_applications':int(unique_output),
        'local_output_path':str(local_output),
        's3_output_key':output_key if uploaded else None,
        'uploaded_to_s3':uploaded,
        'started_at':started.isoformat(timespec='seconds'),
        'finished_at':datetime.now().isoformat(timespec='seconds'),
    }
    logger.info('Completed %s | %s',input_key,result)
    return result,df_output

## 9. Execute selected files

In [ ]:
run_results=[]
last_output_df=None
for i,input_key in enumerate(files_to_process,start=1):
    print(f'\n[{i}/{len(files_to_process)}] {input_key}')
    try:
        result,last_output_df=process_one_file(input_key)
        run_results.append(result)
        print(json.dumps(result,indent=2))
    except Exception:
        logger.exception('FAILED %s',input_key)
        raise  # fail fast; do not silently skip a backfill/delta file
print('\nRun complete')

## 10. Run summary

In [ ]:
df_run_summary=pd.DataFrame(run_results)
display(df_run_summary)
summary_path=LOG_DIR / f'{session_run_stamp}_{RUN_MODE.lower()}_summary.parquet'
df_run_summary.to_parquet(summary_path,index=False,engine='pyarrow')
print('Summary:',summary_path)
print('Manifest:',PROCESSING_MANIFEST_PATH)

## 11. Inspect latest final output, including native-null prediction rows

In [ ]:
if last_output_df is not None:
    display(last_output_df.head(20))
    print(last_output_df.dtypes)
    print('Unique applications:',last_output_df[ID_FIELDS].drop_duplicates().shape[0])
    null_rows=last_output_df[last_output_df['category_id'].isna()]
    print('Null-prediction applications:',null_rows[ID_FIELDS].drop_duplicates().shape[0])
    display(null_rows.head(20))

## Running the two modes

### Initial 10-file run
Set `RUN_MODE = "BACKFILL"`.

- If the input prefix contains exactly the 10 agreed files, leave `BACKFILL_KEYS = []`.
- If the prefix also contains other historical/delta files, populate `BACKFILL_KEYS` explicitly with the 10 source object keys.
- Each file is processed independently and produces its own Parquet output.

### Weekly delta
Set `RUN_MODE = "DELTA"`.

The notebook lists timestamped Parquet files, removes entries already marked `SUCCESS` in the local Ronin manifest, and selects the newest remaining file by the timestamp in the filename. The manifest is updated only after the final Parquet has been successfully uploaded and verified in S3.

### Native nulls
The literal string `"NULL"` is never written. Unclassified/no-text applications carry native Parquet nulls in `category_id`, `score_type`, and `score` while `Taxonomy`, `model_name`, `model_version`, IDs, and run timestamp remain populated.